# NB10 — VLM V2 real end-to-end run

Chạy **VALID thật** theo pipeline:

`frozen scorer → LOO → Recommendation V2 → vlm-evidence-v2 → outfit crops + Top-3 images → Qwen3-VL → validator → Vietnamese renderer → handoff`

Notebook **không dùng synthetic GT/negative_metadata làm input cho VLM**. Bản baseline chỉ dùng item crops, chưa dùng full outfit image.


In [ ]:
from pathlib import Path
import json, subprocess, sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/vlm-recommendation-evidence-v2"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",REPO_URL,str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git","-C",str(REPO_ROOT),"fetch","origin",BRANCH], check=True)
    subprocess.run(["git","-C",str(REPO_ROOT),"checkout",BRANCH], check=True)
    subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only","origin",BRANCH], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(REPO_ROOT/"requirements-vlm.txt")], check=True)

HEAD = subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"], text=True).strip()
print("Branch:", BRANCH)
print("HEAD  :", HEAD)


## Drive paths

Recommendation V2 directory mode đã được viết cho layout thật trên Drive: `ML_Final` + `images`.

Nếu shortcut/folder của bạn có tên khác, chỉ sửa **2 dòng** dưới đây.


In [ ]:
ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
IMAGE_ROOT = Path("/content/drive/MyDrive/images")

assert ARTIFACT_ROOT.is_dir(), f"Không thấy ML_Final: {ARTIFACT_ROOT}"
assert IMAGE_ROOT.is_dir(), f"Không thấy images: {IMAGE_ROOT}"

print("ML_Final:", ARTIFACT_ROOT)
print("images  :", IMAGE_ROOT)


In [ ]:
# Fail fast trước khi download Qwen 4B.
patterns = [
    "test_vlm_evidence_v2.py",
    "test_vlm_prompt_v2.py",
    "test_vlm_validator_v2.py",
    "test_vlm_pipeline_v2.py",
    "test_vlm_explanation.py",
]
for pattern in patterns:
    run = subprocess.run(
        [sys.executable,"-m","unittest","discover","-s","tests","-p",pattern,"-v"],
        cwd=REPO_ROOT, text=True, capture_output=True
    )
    print(run.stdout)
    if run.stderr:
        print(run.stderr)
    if run.returncode != 0:
        raise RuntimeError(f"Tests failed: {pattern}")
print("VLM V1 + V2 TESTS: PASS")


In [ ]:
import torch
from src.diagnosis.loo import diagnose_outfit
from src.recommendation import RecommendationPipeline

assert torch.cuda.is_available(), "Colab: Runtime → Change runtime type → T4 GPU"

rec_pipeline = RecommendationPipeline.load_from_directories(
    REPO_ROOT / "configs" / "recommendation_category_aware_v2.json",
    artifact_root=ARTIFACT_ROOT,
    image_root=IMAGE_ROOT,
    device="cpu",   # giữ GPU trống cho Qwen
)
records = rec_pipeline.artifact_bundle.load_scorer_ready("valid")

print("VALID records:", len(records))
print("Catalog      :", len(rec_pipeline.catalog))
print("Image map    :", rec_pipeline.image_validation)


In [ ]:
# Chọn deterministic VALID negative >=4 items, nhưng KHÔNG đọc negative_metadata/GT.
# Loop cho tới khi có một case chạy đủ LOO + Recommendation + 7 ảnh thật.
sample_row = loo_result = recommendation_result = None
failures = []

for row in sorted(records, key=lambda x: str(x.get("sample_id",""))):
    if row.get("label") != 0 or len(row.get("items", [])) < 4:
        continue
    item_ids = [str(x) for x in row["items"]]
    try:
        embeddings = rec_pipeline.catalog.get_embeddings(item_ids)
        category_ids = [rec_pipeline.metadata.category_id(x) for x in item_ids]
        if any(x is None for x in category_ids):
            continue
        category_ids = [int(x) for x in category_ids]

        loo = diagnose_outfit(
            rec_pipeline.reranker.scorer,
            torch.as_tensor(embeddings, dtype=torch.float32),
            torch.as_tensor(category_ids, dtype=torch.long),
            item_ids=item_ids,
        )
        rec = rec_pipeline.recommend(
            outfit_item_ids=item_ids,
            outfit_embeddings=embeddings,
            outfit_category_ids=category_ids,
            problematic_index=int(loo["problematic_item_index"]),
            loo_result=loo,
            query_id=str(row["sample_id"]),
            source_split=None,
            ground_truth_item_id=None,
        )

        all_needed = item_ids + [x.item_id for x in rec.items]
        image_failures = rec_pipeline.image_resolver.validate_readable(all_needed)
        if image_failures:
            failures.append((row["sample_id"], image_failures))
            continue

        sample_row, loo_result, recommendation_result = row, loo, rec
        break
    except Exception as e:
        failures.append((row.get("sample_id"), f"{type(e).__name__}: {e}"))

assert sample_row is not None, f"Không tìm được case chạy sạch. First failures: {failures[:3]}"

item_ids = [str(x) for x in sample_row["items"]]
coarse_categories = [str(rec_pipeline.metadata.coarse_category(x)) for x in item_ids]

print("Sample        :", sample_row["sample_id"])
print("Items         :", list(zip(item_ids, coarse_categories)))
print("Problematic   :", loo_result["problematic_item_index"], loo_result["problematic_item_id"])
print("Top-3         :", [(x.rank, x.item_id, x.coarse_category) for x in recommendation_result.items])


In [ ]:
# Materialize đúng N outfit images + 3 candidate images.
RUN_DIR = Path("/content/vlm_v2_real") / str(sample_row["sample_id"])
RUN_DIR.mkdir(parents=True, exist_ok=True)

outfit_image_paths = []
for idx, item_id in enumerate(item_ids):
    path = RUN_DIR / f"outfit_{idx:02d}_{item_id}.jpg"
    rec_pipeline.image_resolver.write_selected_image(item_id, path)
    outfit_image_paths.append(path)

recommendation_image_paths = {}
for item in recommendation_result.items:
    path = RUN_DIR / f"recommendation_rank{item.rank}_{item.item_id}.jpg"
    rec_pipeline.image_resolver.write_selected_image(item.item_id, path)
    recommendation_image_paths[item.item_id] = path

print("Outfit images:", len(outfit_image_paths))
print("Rec images   :", len(recommendation_image_paths))


In [ ]:
# Nhìn ảnh trước khi chạy Qwen.
import matplotlib.pyplot as plt
from PIL import Image

display_rows = []
for idx, path in enumerate(outfit_image_paths):
    display_rows.append((f"OUTFIT {idx}\n{item_ids[idx]}\n{coarse_categories[idx]}", path))
for item in recommendation_result.items:
    display_rows.append((f"REC #{item.rank}\n{item.item_id}\n{item.coarse_category}", recommendation_image_paths[item.item_id]))

fig, axes = plt.subplots(1, len(display_rows), figsize=(3*len(display_rows), 4))
for ax, (title, path) in zip(axes, display_rows):
    ax.imshow(Image.open(path))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
from src.vlm import (
    build_vlm_evidence_v2,
    load_vlm_config_v2,
    Qwen3VLBackendV2,
    VLMExplanationPipelineV2,
)

evidence = build_vlm_evidence_v2(
    loo_result,
    recommendation_result,
    sample_id=str(sample_row["sample_id"]),
    item_ids=item_ids,
    coarse_categories=coarse_categories,
)

# Safety check: serialized VLM evidence không được chứa synthetic evaluation answer.
serialized = json.dumps(evidence, ensure_ascii=False)
for forbidden in (
    "negative_metadata","swapped_item_index","target_swapped_item_index",
    "original_item_id","ground_truth_item_id","replacement_item_id",
):
    assert forbidden not in serialized, forbidden

vlm_config = load_vlm_config_v2(
    REPO_ROOT / "configs" / "vlm_qwen3_vl_4b_instruct_v2.json"
)

print("Evidence schema:", evidence["schema_version"])
print("Recommendation:", [(x["rank"], x["item_id"]) for x in evidence["recommendation"]["items"]])


In [ ]:
# REAL QWEN RUN — cell nặng nhất: download/load Qwen3-VL-4B-Instruct rồi inference.
backend = Qwen3VLBackendV2.from_config(vlm_config)
vlm_pipeline = VLMExplanationPipelineV2(backend, vlm_config)

vlm_run = vlm_pipeline.explain(
    evidence,
    outfit_image_paths,
    recommendation_image_paths,
    must_exist=True,
)

print("generation_attempts:", vlm_run["generation_attempts"])
print("\n=== VISUAL ANALYSIS (Qwen, đã qua validator) ===")
print(json.dumps(vlm_run["visual_analysis"], indent=2, ensure_ascii=False))
print("\n=== HANDOFF / FINAL VI ===")
print(json.dumps(vlm_run["handoff"], indent=2, ensure_ascii=False))


In [ ]:
# Lưu full run để review với nhóm.
SAVE_DIR = Path("/content/drive/MyDrive/vlm_v2_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = SAVE_DIR / f"{sample_row['sample_id']}_vlm_v2.json"
SAVE_PATH.write_text(json.dumps(vlm_run, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", SAVE_PATH)
